In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM (GB): 4.29


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [3]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total: {total:,} | Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

count_params(model)

Total: 66,955,010 | Trainable: 66,955,010 (100.00%)


In [4]:
for name, param in model.named_parameters():
    print(f"{name:70s} {tuple(param.shape)}  trainable={param.requires_grad}")

distilbert.embeddings.word_embeddings.weight                           (30522, 768)  trainable=True
distilbert.embeddings.position_embeddings.weight                       (512, 768)  trainable=True
distilbert.embeddings.LayerNorm.weight                                 (768,)  trainable=True
distilbert.embeddings.LayerNorm.bias                                   (768,)  trainable=True
distilbert.transformer.layer.0.attention.q_lin.weight                  (768, 768)  trainable=True
distilbert.transformer.layer.0.attention.q_lin.bias                    (768,)  trainable=True
distilbert.transformer.layer.0.attention.k_lin.weight                  (768, 768)  trainable=True
distilbert.transformer.layer.0.attention.k_lin.bias                    (768,)  trainable=True
distilbert.transformer.layer.0.attention.v_lin.weight                  (768, 768)  trainable=True
distilbert.transformer.layer.0.attention.v_lin.bias                    (768,)  trainable=True
distilbert.transformer.layer.0.attenti

In [5]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
train_ds = dataset["train"].shuffle(seed=42).select(range(4000))
test_ds = dataset["test"].shuffle(seed=42).select(range(1000))

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

print(train_ds)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})


In [6]:
import numpy as np, time
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

def run_experiment(model, name, lr=5e-4, epochs=2):
    args = TrainingArguments(
        output_dir=f"./out_{name}",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        num_train_epochs=epochs,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=test_ds,
        data_collator=collator, compute_metrics=compute_metrics,
    )
    start = time.time()
    trainer.train()
    result = trainer.evaluate()
    print(f"\n[{name}] accuracy={result['eval_accuracy']:.4f} | time={time.time()-start:.0f}s")
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    return result

In [7]:
# Fresh model each experiment
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Freeze the body
for param in model.distilbert.parameters():
    param.requires_grad = False

count_params(model)   # expect ~0.6M trainable (~0.9%)

torch.cuda.reset_peak_memory_stats()
res1 = run_experiment(model, "head_only")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total: 66,955,010 | Trainable: 592,130 (0.88%)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.423307,0.435597,0.802000
2,0.351159,0.398468,0.837000


Training Loss,Validation Loss,Epoch,Accuracy
0.351159,0.398468,2,0.837000



[head_only] accuracy=0.8370 | time=38s
Peak VRAM: 0.47 GB


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 1. Freeze the whole body
for param in model.distilbert.parameters():
    param.requires_grad = False

# 2. Unfreeze the top 2 transformer blocks
for layer in model.distilbert.transformer.layer[4:]:
    for param in layer.parameters():
        param.requires_grad = True

count_params(model)   # expect ~29% trainable

torch.cuda.reset_peak_memory_stats()
res2 = run_experiment(model, "top2_blocks", lr=1e-4)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total: 66,955,010 | Trainable: 14,767,874 (22.06%)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.348833,0.301310,0.873000
2,0.181372,0.310260,0.870000


Training Loss,Validation Loss,Epoch,Accuracy
0.181372,0.310260,2,0.870000



[top2_blocks] accuracy=0.8700 | time=55s
Peak VRAM: 0.76 GB


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Nothing frozen
count_params(model)   # expect 100% trainable

torch.cuda.reset_peak_memory_stats()
res3 = run_experiment(model, "full_finetune", lr=3e-5)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total: 66,955,010 | Trainable: 66,955,010 (100.00%)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.343725,0.297897,0.869000
2,0.139557,0.292013,0.890000


Training Loss,Validation Loss,Epoch,Accuracy
0.139557,0.292013,2,0.890000



[full_finetune] accuracy=0.8900 | time=107s
Peak VRAM: 1.83 GB


In [12]:
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],   # the attention layers we spotted earlier
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

torch.cuda.reset_peak_memory_stats()
res4 = run_experiment(model, "lora", lr=5e-4)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


Epoch,Training Loss,Validation Loss,Accuracy
1,0.340792,0.314763,0.853000
2,0.208387,0.308769,0.869000


Training Loss,Validation Loss,Epoch,Accuracy
0.208387,0.308769,2,0.869000



[lora] accuracy=0.8690 | time=78s
Peak VRAM: 1.08 GB


In [13]:
configs = {
    "lora_r8_qv_4ep":   dict(r=8,  targets=["q_lin", "v_lin"], epochs=4),
    "lora_r16_all_4ep": dict(r=16, targets=["q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2"], epochs=4),
}

results = {}
for name, c in configs.items():
    base = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=c["r"], lora_alpha=2*c["r"],
                     lora_dropout=0.1, target_modules=c["targets"])
    m = get_peft_model(base, cfg)
    m.print_trainable_parameters()
    torch.cuda.reset_peak_memory_stats()
    results[name] = run_experiment(m, name, lr=5e-4, epochs=c["epochs"])

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


Epoch,Training Loss,Validation Loss,Accuracy
1,0.350070,0.311118,0.856000
2,0.223402,0.324456,0.850000
3,0.237943,0.309364,0.877000
4,0.182090,0.322019,0.875000


Training Loss,Validation Loss,Epoch,Accuracy
0.182090,0.322019,4,0.875000



[lora_r8_qv_4ep] accuracy=0.8750 | time=151s
Peak VRAM: 1.36 GB


... (2 outputs left out)

trainable params: 1,919,234 || all params: 68,874,244 || trainable%: 2.7866


Epoch,Training Loss,Validation Loss,Accuracy
1,0.360972,0.389166,0.831000
2,0.196451,0.382532,0.851000
3,0.147635,0.338510,0.881000
4,0.057263,0.492406,0.876000


Training Loss,Validation Loss,Epoch,Accuracy
0.057263,0.492406,4,0.876000



[lora_r16_all_4ep] accuracy=0.8760 | time=225s
Peak VRAM: 1.88 GB
